# Explore Transformer Internals

This notebook digs deeper into the Transformer architecture: tokenization, embeddings, attention patterns across layers, and the causal mask in GPT.

---

In [ ]:
%pip install -q transformers torch matplotlib seaborn numpy

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import (
    BertTokenizer, BertModel,
    GPT2Tokenizer, GPT2LMHeadModel,
)

## 1 — Tokenization: How Transformers See Text

Unlike Session 1's word-level tokenization, Transformers use **subword** tokenization (WordPiece for BERT, BPE for GPT). This means:
- Common words stay whole: "the" → `["the"]`
- Rare words get split: "unhappiness" → `["un", "##happiness"]`
- The model never sees an "unknown" word

In [ ]:
bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')
gpt_tok = GPT2Tokenizer.from_pretrained('gpt2')

sentences = [
    "The cat sat on the mat",
    "Transformers revolutionized natural language processing",
    "supercalifragilisticexpialidocious",
    "COVID-19 vaccination rates",
]

for sent in sentences:
    bert_tokens = bert_tok.tokenize(sent)
    gpt_tokens = gpt_tok.tokenize(sent)
    print(f'\n"{sent}"')
    print(f'  BERT ({len(bert_tokens)} tokens): {bert_tokens}')
    print(f'  GPT2 ({len(gpt_tokens)} tokens): {gpt_tokens}')

## 2 — Token Embeddings + Positional Encodings

Each token gets two vectors added together:
1. **Token embedding** — what the token means (like Word2Vec from Session 1)
2. **Position embedding** — where the token sits in the sequence

In [ ]:
bert_model = BertModel.from_pretrained('bert-base-uncased', output_attentions=True)
bert_model.eval()

# Look at the embedding layers
embeddings = bert_model.embeddings
print(f"Token embedding table: {embeddings.word_embeddings.weight.shape}")
print(f"  → {embeddings.word_embeddings.weight.shape[0]:,} vocab × {embeddings.word_embeddings.weight.shape[1]} dims")
print(f"\nPosition embedding table: {embeddings.position_embeddings.weight.shape}")
print(f"  → {embeddings.position_embeddings.weight.shape[0]} positions × {embeddings.position_embeddings.weight.shape[1]} dims")

In [ ]:
# Visualize position embeddings — nearby positions should be similar
pos_emb = embeddings.position_embeddings.weight.detach().numpy()[:50]

# Cosine similarity between position embeddings
norms = np.linalg.norm(pos_emb, axis=1, keepdims=True)
pos_sim = (pos_emb @ pos_emb.T) / (norms @ norms.T)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pos_sim, cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Position Embedding Similarity (positions 0-49)', fontsize=14)
ax.set_xlabel('Position')
ax.set_ylabel('Position')
plt.tight_layout()
plt.show()

print("Notice: nearby positions have high similarity (warm colors along the diagonal)")

## 3 — Attention Across Layers

Different layers learn different things:
- **Early layers**: local/syntactic patterns
- **Middle layers**: semantic relationships
- **Deep layers**: abstract/task-specific patterns

In [ ]:
sentence = "The trophy didn't fit in the suitcase because it was too big"
inputs = bert_tok(sentence, return_tensors='pt')
tokens = bert_tok.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    outputs = bert_model(**inputs)

# Show attention from "it" at layers 1, 6, and 12
it_idx = tokens.index('it')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, layer_idx in zip(axes, [0, 5, 11]):
    # Average across all heads for this layer
    attn = outputs.attentions[layer_idx][0].mean(dim=0).numpy()
    it_attn = attn[it_idx]
    
    ax.barh(range(len(tokens)), it_attn, color='steelblue')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    ax.invert_yaxis()
    ax.set_title(f'Layer {layer_idx + 1} (avg over heads)', fontsize=12)
    ax.set_xlabel('Attention weight')

plt.suptitle(f'What does "it" attend to at different layers?', fontsize=14)
plt.tight_layout()
plt.show()

## 4 — The Causal Mask in GPT

GPT uses **causal (masked) attention**: each token can only attend to tokens **before** it.
This is what makes it a *decoder* — it can't peek at the future.

In [ ]:
# Visualize the causal mask
seq_len = 8
causal_mask = np.tril(np.ones((seq_len, seq_len)))

example_tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat', '.', '[next?]']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BERT: full attention (encoder)
sns.heatmap(np.ones((seq_len, seq_len)), cmap='Blues', ax=axes[0],
            xticklabels=example_tokens, yticklabels=example_tokens,
            square=True, cbar=False, linewidths=0.5)
axes[0].set_title('BERT (Encoder) — sees everything', fontsize=12)

# GPT: causal attention (decoder)
sns.heatmap(causal_mask, cmap='Blues', ax=axes[1],
            xticklabels=example_tokens, yticklabels=example_tokens,
            square=True, cbar=False, linewidths=0.5)
axes[1].set_title('GPT (Decoder) — can only look left', fontsize=12)

for ax in axes:
    ax.set_xlabel('Attended to')
    ax.set_ylabel('Attending from')

plt.tight_layout()
plt.show()

print('White = can attend, Dark blue = can attend, Empty = masked (cannot see)')
print('GPT\'s lower triangle means each token only sees tokens before it.')

## 5 — Step-by-Step Autoregressive Generation

Let's watch GPT-2 generate text **one token at a time**, showing the probability distribution at each step.

In [ ]:
gpt_model_gen = GPT2LMHeadModel.from_pretrained('gpt2')
gpt_model_gen.eval()

prompt = "The cat sat on"
input_ids = gpt_tok.encode(prompt, return_tensors='pt')

print(f'Starting prompt: "{prompt}"\n')
print(f'{"Step":>4s}  {"Generated":15s}  {"Top 5 candidates (probability)"}' )
print('-' * 70)

generated_ids = input_ids.clone()

for step in range(10):
    with torch.no_grad():
        outputs = gpt_model_gen(generated_ids)
    
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=0)
    top_probs, top_indices = torch.topk(probs, 5)
    
    # Greedy: pick the top token
    next_id = top_indices[0].unsqueeze(0).unsqueeze(0)
    generated_ids = torch.cat([generated_ids, next_id], dim=1)
    
    chosen_token = gpt_tok.decode([top_indices[0]])
    candidates = ', '.join(
        f'{gpt_tok.decode([idx])}({p:.2f})'
        for p, idx in zip(top_probs, top_indices)
    )
    print(f'{step+1:4d}  {chosen_token:15s}  {candidates}')

full_text = gpt_tok.decode(generated_ids[0], skip_special_tokens=True)
print(f'\nFull output: "{full_text}"')

## 6 — Temperature Visualization

Let's see *exactly* how temperature reshapes the probability distribution.

In [ ]:
prompt = "The president of the United States"
inputs = gpt_tok(prompt, return_tensors='pt')

with torch.no_grad():
    outputs = gpt_model_gen(**inputs)

logits = outputs.logits[0, -1, :]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, temp in zip(axes, [0.3, 1.0, 2.0]):
    scaled_probs = torch.softmax(logits / temp, dim=0)
    top_probs, top_indices = torch.topk(scaled_probs, 15)
    
    tokens = [gpt_tok.decode([idx]).strip() for idx in top_indices]
    probs = top_probs.numpy()
    
    ax.barh(range(len(tokens)), probs, color='steelblue')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    ax.invert_yaxis()
    ax.set_title(f'Temperature = {temp}', fontsize=12)
    ax.set_xlabel('Probability')
    ax.set_xlim(0, 1)

plt.suptitle(f'Next token after: "{prompt}"', fontsize=14)
plt.tight_layout()
plt.show()

print('Low temp → one token dominates (safe/repetitive)')
print('High temp → probabilities spread out (creative/unpredictable)')

## 7 — Your Turn!

Experiment freely below.

In [ ]:
# Your experiment here!
# Ideas:
# - Try attention on sentences in different languages
# - Compare "it" resolution in different sentences
# - Generate text with unusual prompts
# - Check what BERT predicts for [MASK] in different contexts